In [1]:
from modules.data import get_historical_data
from modules.database_query import get_price_data

from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import pandas as pd

import pandas_market_calendars as mcal

db password ········


In [2]:
#historical_data = get_historical_data(days = 5, ticker_num = 5, strategy_configs[strategy_name])
historical_data = get_price_data('TSLA')

UndefinedColumn: column "symbol_id" does not exist
LINE 3:                     INNER JOIN symbol on symbol_id = symbol....
                                                 ^
HINT:  Perhaps you meant to reference the column "price_data.symbol".

In [ ]:
historical_data['price_date'] = pd.to_datetime(historical_data['price_date'])
historical_data.head()

In [ ]:
historical_data.iloc[0].price_date

In [ ]:
def reindex_to_market_hours(df, freq='1min', exchange_map=None):
    """
    Reindex OHLCV data to expected trading hours per ticker's exchange calendar.

    Parameters:
    - df: DataFrame with ['price_date', 'ticker', 'open', 'high', 'low', 'close']
    - freq: Frequency for reindexing, e.g., '1min', '5min'
    - exchange_map: Optional dict like {'AAPL': 'NYSE', 'GOOG': 'NASDAQ'}

    Returns:
    - Reindexed DataFrame with missing timestamps as NaNs
    """
    df['price_date'] = pd.to_datetime(df['price_date'])
    all_data = []

    tickers = df['ticker'].unique()
    #exchange_map = exchange_map or {}
    #full_exchange_map = {ticker: exchange_map.get(ticker, infer_exchange(ticker)) for ticker in tickers}

    for ticker, group in df.groupby('ticker'):
        exchange = 'NYSE'
        cal = mcal.get_calendar(exchange)

        start = group['price_date'].min().date()
        end = group['price_date'].max().date()

        sched = cal.schedule(start_date=start, end_date=end)
        trading_minutes = mcal.date_range(schedule=sched, frequency=freq)
        trading_index = pd.DatetimeIndex(trading_minutes)

        group = group.set_index('price_date').sort_index()
        group = group.reindex(trading_index)
        group['ticker'] = ticker

        all_data.append(group.reset_index().rename(columns={'index': 'price_date'}))

    return pd.concat(all_data, ignore_index=True), trading_minutes

In [ ]:
h,tm = reindex_to_market_hours(historical_data)

In [ ]:
tm

In [ ]:
tm[0]

In [ ]:
freq='1min'
exchange = 'NYSE'
cal = mcal.get_calendar(exchange)

start = historical_data['price_date'].min().date()
end = historical_data['price_date'].max().date()

sched = cal.schedule(start_date=start, end_date=end)
trading_minutes = mcal.date_range(schedule=sched, frequency=freq)

In [ ]:
sched